# 2. Model Training with Cost-Sensitive Ensembles
Utilizing Cost-Sensitive Bagging (Random Forest), Boosting (XGBoost), and Decision Trees to handle the severe 96.5% class imbalance.

In [1]:
import pandas as pd
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier # Bagging
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier # Boosting

df = pd.read_csv('data/processed/processed_data.csv')
X = df.drop(columns=['Target'])
y = df['Target']

# Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
os.makedirs('models', exist_ok=True)

# Save encoder
with open('models/label_encoder.pkl', 'wb') as f: pickle.dump(le, f)

# -- Special Approach: Cost-Sensitive Learning --
# Industrial datasets have extreme imbalance (96.5% no failure). 
# We explicitly penalize the model heavily for missing actual failures (False Negatives).
print('Applying Cost-Sensitive Learning to handle severe 96.5% class imbalance...')
sample_weights = compute_sample_weight('balanced', y_train)

# -- Model 1: Cost-Sensitive Decision Tree (Baseline)
dt = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
with open('models/decision_tree.pkl', 'wb') as f: pickle.dump(dt, f)

# -- Model 2: Cost-Sensitive Random Forest (Bagging)
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
with open('models/random_forest.pkl', 'wb') as f: pickle.dump(rf, f)

# -- Model 3: Cost-Sensitive XGBoost (Boosting with GPU fallback)
xgb = XGBClassifier(tree_method='hist', device='cuda', random_state=42)
try:
    xgb.fit(X_train, y_train, sample_weight=sample_weights)
    print('Cost-Sensitive XGBoost trained successfully on GPU!')
except Exception as e:
    print('GPU not available. Falling back to CPU for XGBoost.')
    xgb = XGBClassifier(tree_method='hist', device='cpu', n_jobs=-1, random_state=42)
    xgb.fit(X_train, y_train, sample_weight=sample_weights)

with open('models/xgboost.pkl', 'wb') as f: pickle.dump(xgb, f)

print('\nAll Ensembles trained & saved successfully to ./models')

Applying Cost-Sensitive Learning to handle severe 96.5% class imbalance...


Cost-Sensitive XGBoost trained successfully on GPU!

All Ensembles trained & saved successfully to ./models
